# Voting classifier
Hard and soft voting, weights, and voting over one algorithm with different settings. The interactive decision-surface demo is `app.py` (`python app.py`, then open http://127.0.0.1:8050).

In [ ]:
import numpy as np
import pandas as pd
from sklearn.calibration import CalibratedClassifierCV
from sklearn.datasets import load_iris, make_classification
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_score
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC

## 1. Iris, made harder
Keep only versicolor and virginica, and only the two sepal columns, where the two species overlap a lot.

In [ ]:
iris = load_iris(as_frame=True)
df = iris.frame.rename(columns={"target": "species"})
df.head()

In [ ]:
# Drop setosa (species 0) and keep the two sepal columns plus the label
new_df = df[df["species"] != 0][["sepal length (cm)", "sepal width (cm)", "species"]]
print(new_df.shape)

# Inputs and output come from new_df (not from the full df)
X = new_df.iloc[:, 0:2]
y = new_df.iloc[:, -1]

## 2. Each base model alone (10-fold cross-validation)

In [ ]:
clf1 = LogisticRegression()
clf2 = RandomForestClassifier(random_state=42)
clf3 = KNeighborsClassifier()

# estimators: a list of (name, model) pairs
estimators = [("lr", clf1), ("rf", clf2), ("knn", clf3)]

for name, model in estimators:
    scores = cross_val_score(model, X, y, cv=10, scoring="accuracy")
    print(name, np.round(np.mean(scores), 2))

## 3. Hard and soft voting

In [ ]:
vc = VotingClassifier(estimators=estimators, voting="hard")
print("hard", np.round(np.mean(cross_val_score(vc, X, y, cv=10, scoring="accuracy")), 2))

vc_soft = VotingClassifier(estimators=estimators, voting="soft")
print("soft", np.round(np.mean(cross_val_score(vc_soft, X, y, cv=10, scoring="accuracy")), 2))

## 4. Weights: give some models a bigger say
Try every weight from 1 to 3 for each model (27 combinations), with soft voting.

In [ ]:
results = []
for i in range(1, 4):
    for j in range(1, 4):
        for k in range(1, 4):
            vc = VotingClassifier(estimators=estimators, voting="soft", weights=[i, j, k])
            acc = np.mean(cross_val_score(vc, X, y, cv=10, scoring="accuracy"))
            results.append((i, j, k, round(acc, 3)))
weights = pd.DataFrame(results, columns=["lr", "rf", "knn", "accuracy"]).sort_values("accuracy", ascending=False)
weights.head(8)

## 5. One algorithm, five settings
SVMs with a polynomial kernel of degree 1 to 5, on a synthetic dataset with 20 columns.
Soft voting needs probabilities; in scikit-learn 1.9 an SVC gets them by wrapping it in CalibratedClassifierCV
(the older SVC(probability=True) is deprecated).

In [ ]:
X, y = make_classification(n_samples=1000, n_features=20, n_informative=15, n_redundant=5, random_state=2)

svms = [(f"svm{d}", CalibratedClassifierCV(SVC(kernel="poly", degree=d), ensemble=False)) for d in range(1, 6)]
for name, model in svms:
    print(name, np.round(np.mean(cross_val_score(model, X, y, cv=10, scoring="accuracy")), 3))

In [ ]:
vc = VotingClassifier(estimators=svms, voting="soft")
print("soft voting over the five SVMs", np.round(np.mean(cross_val_score(vc, X, y, cv=10, scoring="accuracy")), 3))

## 6. The decision-surface demo
The same code as `app.py`: three base models on the concentric-circles data, hard against soft voting.

In [ ]:
from app import figure, run

for voting in ["hard", "soft"]:
    results, X_train, y_train = run("concentric circles", ["logistic regression", "Gaussian naive Bayes", "random forest"], voting)
    print(voting, [(t, round(a, 2)) for t, _, a in results])
figure(results, X_train, y_train)